# Masked Neural Network OOF Generation
This notebook trains a PyTorch Neural Network with an input masking property (`mask_prob`) to robustly handle NaN values. The OOF and test predictions are saved for the stacking ensemble.

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')


In [ ]:
print("Loading data...")
train = pd.read_csv('../datasets/train_original.csv')
test = pd.read_csv('../datasets/test.csv')

X = train.drop(['id', 'addicted_label'], axis=1)
y = train['addicted_label']
X_test = test.drop(['id'], axis=1)


In [ ]:
# Proper Ordinal Mappings
stress_mapping = {'Low': 0, 'Medium': 1, 'High': 2, 'Unknown': -1}
impact_mapping = {'No': 0, 'Yes': 1, 'Unknown': -1}

def apply_mappings(df):
    df_out = df.copy()
    df_out['stress_level'] = df_out['stress_level'].map(stress_mapping).fillna(-1).astype(int)
    df_out['academic_work_impact'] = df_out['academic_work_impact'].map(impact_mapping).fillna(-1).astype(int)
    # Gender one-hot
    df_out = pd.get_dummies(df_out, columns=['gender'], dummy_na=True, dtype=int)
    return df_out

X_preprocessed = apply_mappings(X)
X_test_preprocessed = apply_mappings(X_test)

X_test_preprocessed = X_test_preprocessed.reindex(columns=X_preprocessed.columns, fill_value=0)


In [ ]:
# Feature Engineering
def add_features(df):
    df_out = df.copy()
    denom_screen = df_out['daily_screen_time_hours'].replace(0, 0.001)
    denom_notif = df_out['notifications_per_day'].replace(0, 0.001)

    df_out['social_media_ratio'] = df_out['social_media_hours'] / denom_screen
    df_out['gaming_ratio'] = df_out['gaming_hours'] / denom_screen
    df_out['work_study_ratio'] = df_out['work_study_hours'] / denom_screen
    df_out['app_opens_per_hour'] = df_out['app_opens_per_day'] / denom_screen
    df_out['notifications_to_opens_ratio'] = df_out['app_opens_per_day'] / denom_notif
    df_out['sleep_deficit'] = 8.0 - df_out['sleep_hours']
    return df_out

X_preprocessed = add_features(X_preprocessed)
X_test_preprocessed = add_features(X_test_preprocessed)


In [ ]:
# Handle NaNs by replacing with 0 and adding a Masking Dropout
X_preprocessed = X_preprocessed.fillna(0)
X_test_preprocessed = X_test_preprocessed.fillna(0)


In [ ]:
class MaskedNN(nn.Module):
    def __init__(self, input_dim, mask_prob=0.15):
        super().__init__()
        self.input_mask = nn.Dropout(p=mask_prob) # This acts as our mask_prob to manage NaNs
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        x = self.input_mask(x)
        return self.net(x)


In [ ]:
def train_model(X_tr, y_tr, X_val, y_val, mask_prob=0.15, epochs=30, batch_size=256, lr=1e-3):
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    
    train_dataset = TensorDataset(torch.FloatTensor(X_tr_scaled), torch.FloatTensor(y_tr.values))
    val_dataset = TensorDataset(torch.FloatTensor(X_val_scaled), torch.FloatTensor(y_val.values))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    model = MaskedNN(input_dim=X_tr.shape[1], mask_prob=mask_prob)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    
    best_auc = 0
    best_model_state = None
    
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            out = model(X_batch).squeeze()
            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_preds = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                out = model(X_batch).squeeze()
                val_preds.extend(torch.sigmoid(out).numpy())
                
        val_auc = roc_auc_score(y_val, val_preds)
        scheduler.step(val_auc)
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_model_state = model.state_dict().copy()
            
    model.load_state_dict(best_model_state)
    return model, scaler, best_auc


In [ ]:
N_FOLDS = 5
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_train_nn = np.zeros(len(X_preprocessed))
test_preds_nn = np.zeros(len(X_test_preprocessed))

print(f"Starting {N_FOLDS}-Fold CV OOF generation with Masked NN...")

for fold, (train_idx, val_idx) in enumerate(cv.split(X_preprocessed, y)):
    print(f"--- Fold {fold + 1}/{N_FOLDS} ---")
    
    X_tr, y_tr = X_preprocessed.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X_preprocessed.iloc[val_idx], y.iloc[val_idx]
    
    model, scaler, best_auc = train_model(X_tr, y_tr, X_val, y_val, mask_prob=0.15, epochs=40)
    print(f"  NN Fold {fold+1} Best AUC: {best_auc:.5f}")
    
    # Inference on validation set
    model.eval()
    with torch.no_grad():
        X_val_scaled = scaler.transform(X_val)
        out_val = model(torch.FloatTensor(X_val_scaled)).squeeze()
        oof_train_nn[val_idx] = torch.sigmoid(out_val).numpy()
        
        X_test_scaled = scaler.transform(X_test_preprocessed)
        out_test = model(torch.FloatTensor(X_test_scaled)).squeeze()
        test_preds_nn += torch.sigmoid(out_test).numpy() / N_FOLDS
        
print(f"\nOverall Masked NN OOF AUC: {roc_auc_score(y, oof_train_nn):.5f}")


In [ ]:
os.makedirs('../datasets/oof_preds', exist_ok=True)
np.save('../datasets/oof_preds/oof_train_nn_mask.npy', oof_train_nn)
np.save('../datasets/oof_preds/test_preds_nn_mask.npy', test_preds_nn)
print("Saved NN OOF and Test predictions successfully!")
